# PneumoScan AI — Train all models (Google Colab)

Trains **DenseNet121** and **Swin-Tiny** for Stage 1 (Normal vs Pneumonia) and Stage 2 (Bacterial vs Viral) with identical settings — the exact recipe of the deployed Version 1 models — then calibrates, evaluates, exports the lightweight ONNX files and packs everything for download. About 40–70 minutes on a free T4 GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (free), then *Runtime → Run all*,
or run the cells one by one with **Shift + Enter**. Full explanation: `docs/guide/15-train-from-scratch-step-by-step.md`.

> Data use: the Kermany dataset is public (CC BY 4.0). Testing with real patient images requires consent and
> appropriate institutional approval.

## 1. Get the code
Change `REPO_URL` if you use your own fork.

In [ ]:
import os
REPO_URL = "https://github.com/UzumakiAnirudh/Pneumonia-Check.git"   # <- your repository
if not os.path.exists("/content/project"):
    !git clone -q $REPO_URL /content/project
%cd /content/project/training
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download the Kermany dataset (1.2 GB, no account needed)
From the original Mendeley Data release — takes 2–5 minutes.

In [ ]:
!python download_data.py kermany

## 3. Patient-grouped 70/15/15 split
Pools all images, removes 32 exact duplicates, keeps every patient in a single split. Expect **5,824 images, 2,789 patients**.

In [ ]:
!python prepare_data.py --data-root data/raw/chest_xray --out data/splits.csv

## 4. Train the four models
Watch the `val auc` column rise; each model stops early when it stops improving.

In [ ]:
for arch in ["densenet", "swin"]:
    for task in ["stage1", "stage2"]:
        !python train.py --arch $arch --task $task --epochs 10 --patience 4 --warmup-epochs 0.5 --workers 2

## 5. Calibrate confidence (temperature scaling)

In [ ]:
!python calibrate.py --all --workers 2

## 6. Evaluate on the held-out test set
Writes the dashboard data (`backend/metrics/metrics.json`) and report figures (`training/outputs/figures/`).

In [ ]:
!python evaluate.py --workers 2

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("outputs/figures/*stage1*.png")):
    print(f); display(Image(f, width=320))

## 7. Grad-CAM gallery, example images, ONNX export

In [ ]:
for arch in ["densenet", "swin"]:
    !python gradcam_analysis.py --arch $arch --per-category 6 --max-images 400
!python export_samples.py
!python export_onnx.py --verify

## 8. Download the results
Unzip `pneumoscan_trained.zip` into your project folder (it contains `backend/weights`, `backend/metrics`, `backend/samples`), then restart the backend.

In [ ]:
!cd /content/project && zip -qr /content/pneumoscan_trained.zip backend/weights backend/metrics backend/samples training/outputs
from google.colab import files
files.download("/content/pneumoscan_trained.zip")